In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold


PROJECT = Path(r"Z:\Projects\monsoon-postprocessing")

DATA_FILE = (
    PROJECT
    / "data"
    / "processed"
    / "july2018_regime_labeled_dataset.nc"
)

LABEL_FILE = (
    PROJECT
    / "data"
    / "processed"
    / "july2018_regime_labels.csv"
)

OUTPUT_FILE = (
    PROJECT
    / "data"
    / "processed"
    / "july2018_regime_correction_predictions.nc"
)

METRICS_FILE = (
    PROJECT
    / "data"
    / "processed"
    / "july2018_regime_correction_metrics.csv"
)

In [2]:
with xr.open_dataset(DATA_FILE) as ds:
    model_ds = ds.load()

regime_df = pd.read_csv(
    LABEL_FILE,
    parse_dates=["date"]
)

print(model_ds)
print(regime_df["prototype_regime"].value_counts())

<xarray.Dataset> Size: 15MB
Dimensions:         (date: 31, latitude: 129, longitude: 121)
Coordinates:
  * date            (date) datetime64[ns] 248B 2018-07-01 ... 2018-07-31
  * latitude        (latitude) float64 1kB 6.0 6.25 6.5 6.75 ... 37.5 37.75 38.0
  * longitude       (longitude) float64 968B 68.0 68.25 68.5 ... 97.5 97.75 98.0
Data variables:
    gefs_rainfall   (date, latitude, longitude) float32 2MB 1.91 1.21 ... 1.35
    imerg_rainfall  (date, latitude, longitude) float32 2MB nan nan ... nan nan
    forecast_error  (date, latitude, longitude) float32 2MB nan nan ... nan nan
    mslp            (date, latitude, longitude) float32 2MB 1.008e+03 ... 1.0...
    u850            (date, latitude, longitude) float32 2MB 9.131 ... -7.654
    v850            (date, latitude, longitude) float32 2MB 0.6757 ... -1.168
    q850            (date, latitude, longitude) float32 2MB 10.48 ... 21.09
    wind_speed_850  (date, latitude, longitude) float32 2MB 9.156 9.58 ... 7.742
    regime_cod

In [3]:
FEATURES = [
    "central_gefs_rain",
    "regional_min_mslp",
    "regional_mean_mslp",
    "monsoon_q850",
    "monsoon_wind_speed",
    "arabian_sea_u850"
]

X = regime_df[FEATURES]
y = regime_df["prototype_regime"]


def create_classifier():
    return Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=4,
                min_samples_leaf=2,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            )
        )
    ])

In [4]:
raw_values = model_ds[
    "gefs_rainfall"
].values

observed_values = model_ds[
    "imerg_rainfall"
].values

global_corrected = np.full_like(
    raw_values,
    np.nan,
    dtype=np.float32
)

regime_corrected = np.full_like(
    raw_values,
    np.nan,
    dtype=np.float32
)

predicted_regimes = np.empty(
    model_ds.sizes["date"],
    dtype=object
)

fold_records = []

In [5]:
cross_validation = StratifiedKFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)


for fold, (train_indices, test_indices) in enumerate(
    cross_validation.split(X, y),
    start=1
):
    classifier = create_classifier()

    classifier.fit(
        X.iloc[train_indices],
        y.iloc[train_indices]
    )

    fold_predictions = classifier.predict(
        X.iloc[test_indices]
    )

    train_dates = regime_df.iloc[
        train_indices
    ]["date"].to_numpy()

    train_subset = model_ds.sel(
        date=train_dates
    )

    training_errors = (
        train_subset["gefs_rainfall"]
        - train_subset["imerg_rainfall"]
    )

    global_bias = float(
        training_errors.mean(skipna=True)
    )

    regime_biases = {}

    for regime in sorted(y.unique()):
        regime_dates = regime_df.iloc[
            train_indices
        ]

        regime_dates = regime_dates[
            regime_dates["prototype_regime"]
            == regime
        ]["date"].to_numpy()

        if len(regime_dates) == 0:
            regime_biases[regime] = global_bias
            continue

        regime_subset = model_ds.sel(
            date=regime_dates
        )

        regime_errors = (
            regime_subset["gefs_rainfall"]
            - regime_subset["imerg_rainfall"]
        )

        regime_biases[regime] = float(
            regime_errors.mean(skipna=True)
        )

    for local_position, date_index in enumerate(
        test_indices
    ):
        predicted_regime = fold_predictions[
            local_position
        ]

        predicted_regimes[
            date_index
        ] = predicted_regime

        raw_day = raw_values[
            date_index
        ]

        global_corrected[
            date_index
        ] = np.clip(
            raw_day - global_bias,
            0,
            None
        )

        selected_bias = regime_biases.get(
            predicted_regime,
            global_bias
        )

        regime_corrected[
            date_index
        ] = np.clip(
            raw_day - selected_bias,
            0,
            None
        )

        fold_records.append({
            "fold": fold,
            "date": regime_df.iloc[
                date_index
            ]["date"],
            "true_regime": y.iloc[
                date_index
            ],
            "predicted_regime": predicted_regime,
            "global_bias": global_bias,
            "selected_regime_bias": selected_bias
        })


fold_df = pd.DataFrame(
    fold_records
).sort_values("date")

display(fold_df)

,fold,date,true_regime,predicted_regime,global_bias,selected_regime_bias
8,2,2018-07-01,Normal,Break,5.371294,5.463724
0,1,2018-07-02,Break,Normal,5.362532,4.929460
16,3,2018-07-03,Break,Break,5.394408,5.409928
24,4,2018-07-04,Break,Break,5.223474,5.412852
1,1,2018-07-05,Break,Break,5.362532,5.542441
2,1,2018-07-06,Normal,Break,5.362532,5.542441
25,4,2018-07-07,Break,Normal,5.223474,4.816859
26,4,2018-07-08,Normal,Normal,5.223474,4.816859
27,4,2018-07-09,Normal,Normal,5.223474,4.816859
3,1,2018-07-10,Normal,Break,5.362532,5.542441


In [6]:
def calculate_metrics(
    predicted,
    observed
):
    predicted = np.asarray(
        predicted
    ).ravel()

    observed = np.asarray(
        observed
    ).ravel()

    valid = (
        np.isfinite(predicted)
        & np.isfinite(observed)
    )

    predicted = predicted[valid]
    observed = observed[valid]

    errors = predicted - observed

    return {
        "RMSE": np.sqrt(
            np.mean(errors ** 2)
        ),
        "MAE": np.mean(
            np.abs(errors)
        ),
        "Bias": np.mean(errors),
        "Correlation": np.corrcoef(
            predicted,
            observed
        )[0, 1]
    }


prediction_options = {
    "Raw GEFS": raw_values,
    "Global correction": global_corrected,
    "Regime-aware correction": regime_corrected
}


overall_rows = []

for model_name, prediction in prediction_options.items():
    metrics = calculate_metrics(
        prediction,
        observed_values
    )

    overall_rows.append({
        "model": model_name,
        **metrics
    })


overall_metrics_df = pd.DataFrame(
    overall_rows
).sort_values("RMSE")

overall_metrics_df

,model,RMSE,MAE,Bias,Correlation
2,Regime-aware correction,19.283310,9.319411,1.775371,0.418059
1,Global correction,19.293255,9.326070,1.795499,0.418268
0,Raw GEFS,20.383118,10.980301,5.336696,0.428161


In [7]:
regime_rows = []


for regime in sorted(y.unique()):
    date_mask = (
        regime_df["prototype_regime"]
        == regime
    ).to_numpy()

    for model_name, prediction in prediction_options.items():
        metrics = calculate_metrics(
            prediction[date_mask],
            observed_values[date_mask]
        )

        regime_rows.append({
            "regime": regime,
            "model": model_name,
            "dates": int(date_mask.sum()),
            **metrics
        })


regime_metrics_df = pd.DataFrame(
    regime_rows
)

display(
    regime_metrics_df.sort_values(
        ["regime", "RMSE"]
    )
)

,regime,model,dates,RMSE,MAE,Bias,Correlation
1,Active,Global correction,4,20.835585,10.068924,1.558575,0.505343
2,Active,Regime-aware correction,4,20.862535,10.109785,1.663127,0.505680
0,Active,Raw GEFS,4,21.823767,11.626359,5.080247,0.510436
4,Break,Global correction,9,17.198219,8.376396,1.885100,0.445747
5,Break,Regime-aware correction,9,17.222940,8.402975,1.944143,0.445636
3,Break,Raw GEFS,9,18.403837,10.193403,5.458880,0.453313
8,Low/Depression,Regime-aware correction,7,22.507439,10.680885,2.088520,0.357295
7,Low/Depression,Global correction,7,22.604618,10.802831,2.402603,0.358706
6,Low/Depression,Raw GEFS,7,23.714779,12.424675,6.058847,0.369575
10,Normal,Global correction,11,17.999607,8.893189,1.422004,0.395876


In [8]:
classification_accuracy = np.mean(
    fold_df["true_regime"]
    == fold_df["predicted_regime"]
)

print(
    "Out-of-fold regime accuracy:",
    round(classification_accuracy, 3)
)


confusion_table = pd.crosstab(
    fold_df["true_regime"],
    fold_df["predicted_regime"],
    rownames=["Actual"],
    colnames=["Predicted"]
)

confusion_table

Out-of-fold regime accuracy: 0.548


Predicted,Active,Break,Low/Depression,Normal
Actual,,,,
Active,3,1,0,0
Break,1,5,0,3
Low/Depression,1,0,6,0
Normal,1,7,0,3


In [9]:
prediction_ds = xr.Dataset({
    "raw_gefs_rainfall": (
        model_ds["gefs_rainfall"]
    ),

    "global_corrected_rainfall": (
        (
            model_ds["gefs_rainfall"].dims
        ),
        global_corrected
    ),

    "regime_corrected_rainfall": (
        (
            model_ds["gefs_rainfall"].dims
        ),
        regime_corrected
    ),

    "imerg_rainfall": (
        model_ds["imerg_rainfall"]
    )
})

prediction_ds.attrs = {
    "evaluation": "four-fold out-of-fold",
    "regime_classifier": "Random Forest",
    "correction": "regime-specific additive bias",
    "limitations": "Prototype based on 31 dates"
}

prediction_ds.to_netcdf(
    OUTPUT_FILE,
    mode="w",
    engine="netcdf4"
)

print("Saved:", OUTPUT_FILE.exists())
print("Location:", OUTPUT_FILE)

Saved: True
Location: Z:\Projects\monsoon-postprocessing\data\processed\july2018_regime_correction_predictions.nc


In [10]:
overall_output = overall_metrics_df.copy()
overall_output["scope"] = "overall"
overall_output["regime"] = "All"

regime_output = regime_metrics_df.copy()
regime_output["scope"] = "per-regime"

all_metrics = pd.concat(
    [
        overall_output,
        regime_output
    ],
    ignore_index=True,
    sort=False
)

all_metrics.to_csv(
    METRICS_FILE,
    index=False
)

print("Saved:", METRICS_FILE.exists())

Saved: True


In [11]:
display(overall_metrics_df)
display(regime_metrics_df)
print("Classifier accuracy:", classification_accuracy)

,model,RMSE,MAE,Bias,Correlation
2,Regime-aware correction,19.283310,9.319411,1.775371,0.418059
1,Global correction,19.293255,9.326070,1.795499,0.418268
0,Raw GEFS,20.383118,10.980301,5.336696,0.428161


,regime,model,dates,RMSE,MAE,Bias,Correlation
0,Active,Raw GEFS,4,21.823767,11.626359,5.080247,0.510436
1,Active,Global correction,4,20.835585,10.068924,1.558575,0.505343
2,Active,Regime-aware correction,4,20.862535,10.109785,1.663127,0.505680
3,Break,Raw GEFS,9,18.403837,10.193403,5.458880,0.453313
4,Break,Global correction,9,17.198219,8.376396,1.885100,0.445747
5,Break,Regime-aware correction,9,17.222940,8.402975,1.944143,0.445636
6,Low/Depression,Raw GEFS,7,23.714779,12.424675,6.058847,0.369575
7,Low/Depression,Global correction,7,22.604618,10.802831,2.402603,0.358706
8,Low/Depression,Regime-aware correction,7,22.507439,10.680885,2.088520,0.357295
9,Normal,Raw GEFS,11,19.044117,10.470049,4.870430,0.409765


Classifier accuracy: 0.5483870967741935
